In [1]:
from bs4 import BeautifulSoup
import requests

# TODO investigate rewriting this notebook using Pint.

In [2]:
url = "https://www.kingarthurbaking.com/learn/ingredient-weight-chart"
# This is the data source used for ingredient densities ^^^^^

In [3]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

In [4]:
response = requests.get(url, headers = headers)

In [5]:
soup = BeautifulSoup(response.content, "html.parser")

In [6]:
table = soup.find("table")

In [7]:
import re

assert table is not None

data = []

headers = [th.text.strip() for th in table.find_all("th")]

assert len(headers) == 4

for row in table.find_all("tr") [1:]: # skipping header row
    cells = row.find_all("td")

    if len(cells) > 0:
        data.append([re.sub(r"[^\x20-\x7E]", "", cell.text.strip()) for cell in cells])

In [8]:
import pandas as pd

data_df = pd.DataFrame(data, columns = headers)

In [9]:
data_df

,Ingredient,Volume,Ounces,Grams
0,'00' Pizza Flour,1 cup,4,116
1,Agave syrup,1/4 cup,3,84
2,All-Purpose Baking Mix,1 cup,4 1/4,120
3,All-Purpose Flour,1 cup,4 1/4,120
4,Almond butter,1/4 cup,2 1/3,68
...,...,...,...,...
320,Yeast (instant),2 1/4 teaspoons,1/4,7
321,Yeast (instant),1 tablespoon,1/3,9
322,Yogurt,1 cup,8,227
323,Yuletide Cheer Fruit Blend,1 cup,4 1/2,130


In [10]:
data_df = data_df.drop(columns = ["Ounces"])

In [11]:
# converting the volume column into cm^3 (ml)
from fractions import Fraction

volume_vals = data_df ["Volume"].apply(lambda vol: vol.split(" ") [:3])

# convert quantities into floats
volume_vals = volume_vals.apply(lambda vol: (float(Fraction(vol [0])), vol [1], vol [2]) if len(vol) == 3 else (float(Fraction(vol [0])), vol [1], None))

In [12]:
units = set()

for quantity, extra_quantity_or_unit, _ in volume_vals:
    assert isinstance(quantity, float)

    units.add(extra_quantity_or_unit)

print(units)

{'teaspoon', 'large', 'tablespoon', 'tablespoons', 'teaspoons', '1/4', 'cups', 'cup'}


In [13]:
# volume units in cm^3 (ml)
VOLUME_UNITS = {
    "cup": 236.588,
    "tbsp": 14.7868,
    "tsp": 4.92892,
    "garlic": 90 # for "1 large garlic head", edge case 
}

In [14]:
UNIT_MAP = {
    "cup": "cup", "cups": "cup",
    "tablespoon": "tbsp", "tablespoons": "tbsp",
    "teaspoon": "tsp", "teaspoons": "tsp",
    "large": "garlic" # for "1 large garlic head", edge case
}

In [ ]:
def calculate_volume(quantity, unit, possible_unit = None):
    if unit in UNIT_MAP:
        return quantity * VOLUME_UNITS [UNIT_MAP [unit]]

    # mixed number edge case, e.g. "1 1/4 teaspoon"
    return calculate_volume(quantity + float(Fraction(unit)), possible_unit)

final_volume_vals = [calculate_volume(q, u, p) for q, u, p in volume_vals]


In [16]:
data_df ["Volume"] = final_volume_vals

In [17]:
# some rows have their grams in "x to y" format, so we take the average of x and y as the grams value for those rows

cleaned_grams = []

for grams in data_df ["Grams"]:
    if "to" in grams:
        x, y = grams.split("to")
        cleaned_grams.append((float(x) + float(y)) / 2)
    else:
        cleaned_grams.append(float(grams))

data_df ["Grams"] = cleaned_grams

In [18]:
data_df ["density (g/cm^3)"] = data_df ["Grams"].astype(float) / data_df ["Volume"].astype(float)

In [19]:
data_df.rename(columns = {"Ingredient": "ingredient", "Grams": "grams", "Volume": "volume (cm^3 or ml)"}, inplace = True)

In [20]:
data_df.to_csv("ingredient_densities.csv", index = False)